In [1]:
%pip install h3 folium

In [2]:
import pandas as pd
import numpy as np
import h3, folium, json, tempfile
import notebookutils
from pathlib import Path

# ============================================================
# PARAMÈTRES
# ============================================================
RES_H3   = 5      # même résolution que la carte mobilité (hexagones ~250 km²)
POP_MIN  = 50     # en dessous : hexagone grisé (statistiques peu fiables)
TERRITOIRE = "Métropole"

SRC  = "Files/silver/insee_population/carreaux_revenus_gps.parquet"
DEST = "Files/gold/insee_revenus"

work = Path(tempfile.mkdtemp())

# ============================================================
# 1. LECTURE DE LA SILVER (colonnes utiles uniquement)
# ============================================================
cols = ["latitude", "longitude", "territoire", "Ind", "ind_snv", "men",
        "men_pauv", "ind_65_79", "ind_80p", "log_soc"]
notebookutils.fs.cp(SRC, f"file:{work / 'silver.parquet'}")
df = pd.read_parquet(work / "silver.parquet", columns=cols)
df = df[df["territoire"] == TERRITOIRE]
print(f"{len(df):,} carreaux ({TERRITOIRE})")

# ============================================================
# 2. RATTACHEMENT DE CHAQUE CARREAU À UN HEXAGONE H3
# ============================================================
if hasattr(h3, "latlng_to_cell"):          # h3 v4
    to_cell, to_boundary = h3.latlng_to_cell, h3.cell_to_boundary
else:                                       # h3 v3
    to_cell, to_boundary = h3.geo_to_h3, h3.h3_to_geo_boundary

print("Calcul des hexagones H3 (~20 s)...")
df["h3"] = [to_cell(la, lo, RES_H3)
            for la, lo in zip(df["latitude"].values, df["longitude"].values)]

# ============================================================
# 3. AGRÉGATION : on additionne d'abord, on divise ensuite
# ============================================================
hexa = df.groupby("h3").agg(
    population=("Ind", "sum"),
    somme_nv=("ind_snv", "sum"),
    menages=("men", "sum"),
    menages_pauvres=("men_pauv", "sum"),
    pop_65_79=("ind_65_79", "sum"),
    pop_80p=("ind_80p", "sum"),
    log_sociaux=("log_soc", "sum"),
).reset_index()

def div(a, b):
    return np.where(b > 0, a / b, np.nan)

hexa["niveau_vie_moyen"] = div(hexa["somme_nv"], hexa["population"])
hexa["taux_pauvrete"]    = div(hexa["menages_pauvres"], hexa["menages"])
hexa["part_65_plus"]     = div(hexa["pop_65_79"] + hexa["pop_80p"], hexa["population"])
hexa["part_log_sociaux"] = div(hexa["log_sociaux"], hexa["menages"])

fiable = hexa["population"] >= POP_MIN
for c in ["niveau_vie_moyen", "taux_pauvrete", "part_65_plus", "part_log_sociaux"]:
    hexa.loc[~fiable, c] = np.nan

print(f"{len(hexa):,} hexagones | {fiable.sum():,} avec au moins {POP_MIN} habitants")

# ============================================================
# 4. CLASSES DE COULEUR (seuils fixes ou quartiles)
# ============================================================
ROUGE_VERT = ["#b91c1c", "#ea580c", "#eab308", "#16a34a"]   # palette de ta carte mobilité
VIOLETS    = ["#ede9fe", "#c4b5fd", "#8b5cf6", "#5b21b6"]   # neutre (âge ≠ bien/mal)
GRIS = "#9ca3af"

def classer(serie, palette, fmt, bornes=None):
    if bornes is None:   # quartiles par défaut
        bornes = np.unique(np.nanquantile(serie.dropna(), [0, .25, .5, .75, 1]))
    bornes = np.array(bornes, dtype=float)
    idx = pd.cut(serie, bins=bornes, labels=False, include_lowest=True)
    couleurs = [GRIS if pd.isna(i) else palette[int(i)] for i in idx]
    legende = [(palette[i], f"{fmt(bornes[i])} – {fmt(bornes[i + 1])}")
               for i in range(len(bornes) - 1)]
    legende[0]  = (palette[0],  f"moins de {fmt(bornes[1])}")
    legende[-1] = (palette[len(legende) - 1], f"plus de {fmt(bornes[-2])}")
    return couleurs, legende

euros = lambda v: f"{v:,.0f} €".replace(",", " ")
pct   = lambda v: f"{v * 100:.0f} %"

COUCHES = {
    #                          colonne              palette           format  bornes (None = quartiles)
    "Niveau de vie moyen"    : ("niveau_vie_moyen", ROUGE_VERT,       euros,  [0, 20000, 23000, 26000, np.inf]),
    "Taux de pauvreté"       : ("taux_pauvrete",    ROUGE_VERT[::-1], pct,    [0, .10, .15, .20, 1]),  # pauvreté forte = rouge
    "Part des 65 ans et +"   : ("part_65_plus",     VIOLETS,          pct,    None),
    "Part logements sociaux" : ("part_log_sociaux", VIOLETS,          pct,    None),
}

legendes = {}
for nom, (col, pal, fmt, bornes) in COUCHES.items():
    hexa[f"couleur_{col}"], legendes[nom] = classer(hexa[col], pal, fmt, bornes)

# ============================================================
# 5. GÉOMÉTRIES + INFOBULLES
# ============================================================
def polygone(cell):
    ring = [[lng, lat] for lat, lng in to_boundary(cell)]
    return [ring + [ring[0]]]

def txt(v, fmt):
    return "n.d." if pd.isna(v) else fmt(v)

hexa["popup"] = [
    f"<b>H3 {r.h3}</b><br>"
    f"Population : {r.population:,.0f}<br>".replace(",", " ") +
    f"Niveau de vie moyen : {txt(r.niveau_vie_moyen, euros)} /an<br>"
    f"Ménages pauvres : {txt(r.taux_pauvrete, pct)}<br>"
    f"65 ans et + : {txt(r.part_65_plus, pct)}<br>"
    f"Logements sociaux : {txt(r.part_log_sociaux, pct)}<br>"
    f"<i>Source : INSEE Filosofi 2021, carreaux 200 m</i>"
    for r in hexa.itertuples()
]

def geojson(col):
    return {"type": "FeatureCollection", "features": [
        {"type": "Feature",
         "geometry": {"type": "Polygon", "coordinates": polygone(r["h3"])},
         "properties": {"couleur": r[f"couleur_{col}"], "popup": r["popup"]}}
        for _, r in hexa.iterrows()]}

# ============================================================
# 6. CARTE FOLIUM
# ============================================================
m = folium.Map(location=[46.6, 2.2], zoom_start=6, tiles="openstreetmap")

for i, (nom, (col, _, _, _)) in enumerate(COUCHES.items()):
    folium.GeoJson(
        geojson(col),
        name=nom,
        show=(i == 0),       # seule la 1re couche est affichée au départ
        style_function=lambda f: {"fillColor": f["properties"]["couleur"],
                                  "color": f["properties"]["couleur"],
                                  "weight": 0.6, "fillOpacity": 0.52},
        popup=folium.GeoJsonPopup(fields=["popup"], labels=False, max_width=320),
    ).add_to(m)

folium.LayerControl(position="topright", collapsed=False).add_to(m)

# Légende
blocs = ""
for nom, items in legendes.items():
    lignes = "".join(
        f'<div><span style="display:inline-block;width:12px;height:12px;'
        f'background:{c};margin-right:6px;opacity:.8"></span>{t}</div>'
        for c, t in items)
    blocs += f'<div style="margin-top:6px"><b>{nom}</b>{lignes}</div>'
blocs += (f'<div style="margin-top:6px"><span style="display:inline-block;width:12px;'
          f'height:12px;background:{GRIS};margin-right:6px"></span>Moins de {POP_MIN} hab.</div>')
m.get_root().html.add_child(folium.Element(
    '<div style="position:fixed;bottom:20px;left:10px;z-index:9999;background:white;'
    'padding:8px 10px;border-radius:6px;box-shadow:0 1px 4px rgba(0,0,0,.3);'
    f'font-size:12px;max-height:60vh;overflow:auto">{blocs}</div>'))

# ============================================================
# 7. SAUVEGARDE (carte HTML + table Gold + copie téléchargeable)
# ============================================================
notebookutils.fs.mkdirs(DEST)
for nom_fichier, ecrire in [
    ("carte_revenus_h3_metropole.html", lambda p: m.save(str(p))),
    ("revenus_h3_metropole.parquet",
     lambda p: hexa.drop(columns=["popup"] + [c for c in hexa if c.startswith("couleur_")])
                   .to_parquet(p, index=False)),
]:
    local = work / nom_fichier
    ecrire(local)
    cible = f"{DEST}/{nom_fichier}"
    if notebookutils.fs.exists(cible):
        notebookutils.fs.rm(cible)
    notebookutils.fs.cp(f"file:{local}", cible)
    print("Enregistré :", cible)

# Copie dans Ressources > Intégré pour la télécharger en un clic
Path("builtin").mkdir(exist_ok=True)
m.save("./builtin/carte_revenus_h3_metropole.html")
print("Carte prête : Ressources > Intégré > … > Télécharger")